# Ejercicio 1 — Carga y armonización de datos

In [ ]:
import os
import re
from functools import reduce

import openpyxl
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, LongType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab7")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

RAW_DIR = "../working_dir/raw"
PARQUET_DIR = "../working_dir/parquet"
STAGING_DIR = f"{PARQUET_DIR}/staging"

print(spark.version)
print(sorted(os.listdir(RAW_DIR)))

## 1.1 Identificación del período de cada archivo

In [ ]:
# Metadatos de cada archivo: el período sale del archivo, no de la columna TRIMESTRE
ARCHIVOS = [
    {"archivo": "Personas_ENEIC_T1_2025.xlsx",                "periodo": "2025T1", "anio": 2025, "trimestre": 1},
    {"archivo": "Personas-ENEIC-T2-2025.xlsx",                "periodo": "2025T2", "anio": 2025, "trimestre": 2},
    {"archivo": "Base-de-datos-Personas-ENEIC-III-2025.xlsx", "periodo": "2025T3", "anio": 2025, "trimestre": 3},
    {"archivo": "Base-de-datos-Personas-ENEIC-IV-2025.xlsx",  "periodo": "2025T4", "anio": 2025, "trimestre": 4},
    {"archivo": "Base-de-datos-Personas-ENEIC-I-2026.xlsx",   "periodo": "2026T1", "anio": 2026, "trimestre": 1},
]
pd.DataFrame(ARCHIVOS)

## 1.2 Selección de columnas y tipos

In [ ]:
# Columna original -> nombre analítico
COLUMNAS = {
    "P05D01":      "salario_mensual",
    "P02A03":      "edad",
    "P05C07A":     "antiguedad_anios",
    "P05C07B":     "antiguedad_meses",
    "P05H01A":     "horas_semanales",
    "P03A03A":     "nivel_educativo",
    "P05C16":      "categoria_ocupacional",
    "DOMINIO":     "dominio",
    "OCUPADOS":    "ocupado",
    "NUM_HOGAR":   "NUM_HOGAR",
    "NUM_PERSONA": "NUM_PERSONA",
    "FACTOR":      "FACTOR",
    "ANIO":        "ANIO",
    "TRIMESTRE":   "TRIMESTRE",
}

NUMERICAS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses",
             "horas_semanales", "FACTOR"]
CATEGORICAS = ["nivel_educativo", "categoria_ocupacional", "dominio", "ocupado"]
ENTERAS = {"NUM_HOGAR": LongType(), "NUM_PERSONA": IntegerType(),
           "ANIO": IntegerType(), "TRIMESTRE": IntegerType()}

ESQUEMA = StructType(
    [StructField("archivo_origen", StringType(), False),
     StructField("periodo_archivo", StringType(), False),
     StructField("anio_archivo", IntegerType(), False),
     StructField("trimestre_calendario", IntegerType(), False)]
    + [StructField(c, ENTERAS[c], True) for c in ENTERAS]
    + [StructField(c, StringType(), True) for c in CATEGORICAS]
    + [StructField(c, DoubleType(), True) for c in NUMERICAS]
)
ORDEN_COLUMNAS = [f.name for f in ESQUEMA.fields]

## 1.3 Homologación de códigos

In [ ]:
def normalizar_codigo(valor):
    """Convierte un código a texto sin decimales ('5', 5, 5.0 -> '5'). Vacíos -> None."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    if isinstance(valor, (int, float)):
        return str(int(valor)) if float(valor).is_integer() else str(valor)
    texto = str(valor).strip()
    if texto == "":
        return None
    if re.fullmatch(r"-?\d+\.0+", texto):   # '5.0' -> '5'
        texto = texto.split(".")[0]
    return texto  # códigos no numéricos se conservan para validarlos contra el diccionario


def a_numero(serie):
    """Convierte a float; lo que no sea numérico queda como NaN (se contabiliza después)."""
    return pd.to_numeric(serie.map(lambda v: v.strip() if isinstance(v, str) else v),
                         errors="coerce")


def a_valor_spark(v):
    """pandas usa NaN para faltantes; Spark necesita None para que queden como null."""
    if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA:
        return None
    return v

## 1.4 Carga individual de cada archivo

In [ ]:
def contar_columnas_originales(ruta):
    wb = openpyxl.load_workbook(ruta, read_only=True)
    ws = wb[wb.sheetnames[0]]
    n = len(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
    wb.close()
    return n


def cargar_archivo(meta):
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    pdf = pd.read_excel(ruta, usecols=list(COLUMNAS), dtype=object)
    pdf = pdf.rename(columns=COLUMNAS)

    for c in NUMERICAS:
        pdf[c] = a_numero(pdf[c]).astype(object)
    for c in CATEGORICAS:
        pdf[c] = pdf[c].map(normalizar_codigo)
    for c in ENTERAS:
        pdf[c] = a_numero(pdf[c]).map(lambda v: None if pd.isna(v) else int(v))

    pdf["archivo_origen"] = meta["archivo"]
    pdf["periodo_archivo"] = meta["periodo"]
    pdf["anio_archivo"] = meta["anio"]
    pdf["trimestre_calendario"] = meta["trimestre"]

    filas = [[a_valor_spark(v) for v in fila]
             for fila in pdf[ORDEN_COLUMNAS].itertuples(index=False, name=None)]
    sdf = spark.createDataFrame(filas, schema=ESQUEMA)

    salida = f"{STAGING_DIR}/{meta['periodo']}"
    sdf.write.mode("overwrite").parquet(salida)
    del pdf, filas
    return spark.read.parquet(salida)


resumen_carga = []
bases = {}
for meta in ARCHIVOS:
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    n_cols = contar_columnas_originales(ruta)
    sdf = cargar_archivo(meta)
    bases[meta["periodo"]] = sdf
    resumen_carga.append({"periodo_archivo": meta["periodo"], "archivo_origen": meta["archivo"],
                          "columnas_originales": n_cols, "registros_originales": sdf.count()})
    print(f"{meta['periodo']}: listo")

pd.DataFrame(resumen_carga)

Los conteos de registros y columnas coinciden con los del enunciado: 51,588 / 51,167 / 51,583 / 49,338 / 49,843 registros, y 270 columnas en todos los archivos menos en IV de 2025, que tiene 302. Por esa diferencia de estructura los archivos se unen **por nombre de columna** (`unionByName`) y no por posición.

## 1.5 Unión de los archivos de 2025

In [ ]:
periodos_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]
df_2025 = reduce(lambda a, b: a.unionByName(b), [bases[p] for p in periodos_2025])
df_2026 = bases["2026T1"]

print("Registros 2025:", df_2025.count())
print("Registros 2026:", df_2026.count())
df_2025.printSchema()

### Verificación de la procedencia y de la columna `TRIMESTRE`

La tabla cruza el período asignado según el archivo con el valor original de `TRIMESTRE`. Se ve que la columna original no coincide con el trimestre calendario y que II de 2025 tiene valores mezclados (3 y, en 175 registros, 2). Si se hubiera usado `TRIMESTRE - 1`, esos 175 registros habrían quedado mal asignados a 2025T1.

In [ ]:
(df_2025.unionByName(df_2026)
    .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE")
    .count()
    .orderBy("periodo_archivo", "TRIMESTRE")
    .show(truncate=False))

### Verificación de la homologación de códigos

Después de normalizar, los códigos categóricos tienen la misma representación en todos los archivos, incluido II de 2025, que venía como texto.

In [ ]:
for c in ["categoria_ocupacional", "dominio", "ocupado"]:
    print(f"== {c} ==")
    (df_2025.unionByName(df_2026)
        .groupBy("periodo_archivo").pivot(c).count()
        .orderBy("periodo_archivo")
        .show(truncate=False))

### Esquema y cinco registros de las columnas seleccionadas

In [ ]:
df_2025.show(5, truncate=False)